# Bag-of-Words GRPO

In [ ]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.grpo import BagOfWordsGRPOConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:1")

## Configure

In [ ]:
config = BagOfWordsGRPOConfig.get_canonical(
    dataset="homoskedastic",
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-grpo-example",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
    num_rollouts_per_sample=4,
    gaussian_stdev=1.0,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")
print(f"Rollouts/sample: {config.num_rollouts_per_sample}")
print(f"Policy stdev:    {config.gaussian_stdev}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

In [ ]:
state.run_training()

## Results

In [ ]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

In [ ]:
analysis = BagOfWordsAnalysisConfig.from_grouped({"example": [(0, config.study_folder)]})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])


In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()